# HASI-Net: Heterogeneous Adaptive Spatio-temporal Informer Network
## Forecasting women-centric crimes on aggregated district data — Madhya Pradesh

**Complete deployment notebook.** End-to-end pipeline:

1. Download real public datasets (NCRB district women-crime tables, Census 2011
   socioeconomic features, MP district boundaries, Chicago benchmark).
2. Build the heterogeneous graph (geographic + socioeconomic + learnable adaptive).
3. PSO-tune hyperparameters (adaptive inertia, multi-swarm).
4. Train HASI-Net + baselines (HA, LSTM, ST-GCN, Informer-only).
5. Evaluate (MAE, RMSE, RMSLE, WAPE, R^2, CSI) and generate publication figures.

This notebook is the **reproducible companion** to the paper. Re-running it
top-to-bottom regenerates every result table and figure cited there.

> Architecture and novel contributions are in `src/hasi_net/`. The notebook is
> a thin orchestration layer over that package.


## 0. Environment — Google Colab (T4/V100 GPU)

* Runtime type: **GPU** (Colab Free gives a T4; Pro gives V100/A100).
* Results/models are persisted to Google Drive so they survive session resets.
* The repo's `src/hasi_net/` package is put on the path automatically; if you
  are running without the repo, a setup cell clones it.


In [ ]:
# Install dependencies (Colab GPU runtime).
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for p in ["torch", "numpy", "pandas", "requests", "matplotlib",
          "pyarrow", "geopandas", "shapely"]:
    try:
        __import__(p)
    except Exception:
        pip(p)
print("deps ready")


In [ ]:
# Mount Drive for persistence and put the package on the path.
import sys, pathlib, torch, numpy as np, os
from google.colab import drive
drive.mount("/content/drive")

# If the repo is in Drive, point REPO there; otherwise clone it.
REPO = pathlib.Path("/content/spatio-temporal-crime")
if not REPO.exists():
    REPO = pathlib.Path("/content/drive/MyDrive/spatio-temporal-crime")
if not (REPO / "src" / "hasi_net").exists():
    import subprocess
    subprocess.check_call(["git", "clone", "--depth", "1",
        "https://github.com/<your-user>/spatio-temporal-crime", str(REPO)])
sys.path.insert(0, str(REPO / "src"))

from hasi_net import Config, select_device, set_seed, viz
from hasi_net.experiment import run_experiment
viz.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = select_device("cuda")
set_seed(42)
print("Repo:", REPO)
print("Device:", DEVICE)


In [ ]:
# Colab-tuned configuration (more compute -> larger search + epochs).
cfg = Config(
    device="cuda",
    epochs=120, batch_size=64, lr=1e-3,
    hidden_dim=96, n_graph_layers=2, n_attn_heads=4,
    lookback=4, horizon=2,
    loss_type="log1p_mse", focal_gamma=1.5,
    pso_enabled=True, pso_swarms=4, pso_particles=12, pso_iters=8,
    use_chicago_benchmark=True,
)
print("Config:"); print(cfg.to_dict())


## 1. Data acquisition & preview
Datasets are downloaded (cached under `data/`) on first run. The MP panel uses
the contiguous, fully-real NCRB 2001-2014 district women-crime block (7 IPC
categories) and merges Census 2011 socioeconomic node features. The 2015-2016
and 2017-2022 NCRB releases are not reliably machine-readable, so they are
excluded rather than forward-filled.


In [ ]:
from hasi_net.data import build_mp_panel, build_graphs
panel = build_mp_panel(cfg)
print("Counts tensor [T, N, C]:", panel.counts.shape)
print("Districts (N):", len(panel.districts), "— e.g.", panel.districts[:5])
print("Years (T):", panel.years[0], "->", panel.years[-1], f"({len(panel.years)} steps)")
print("Categories (C):", panel.categories)
print("Socioeconomic features [N, F]:", panel.node_feats.shape)
print("Source:", panel.meta["source"])


## 2. Heterogeneous graph
* `A_geo` — rook contiguity from the MP district boundary GeoJSON.
* `A_socio` — kNN cosine similarity on census features.
* `A_adaptive` — learned at training time (inside the model).


In [ ]:
import numpy as np
A_geo, A_socio = build_graphs(panel.districts, panel.node_feats, cfg)
print("A_geo density:", round(A_geo.mean(), 4), " A_socio density:", round(A_socio.mean(), 4))
print("A_geo edges:", int((A_geo > 0).sum() // 2),
      " A_socio edges:", int((A_socio > 0).sum() // 2))


## 3. Full experiment — Madhya Pradesh (target region)
`run_experiment` does: assemble panel + graphs -> adaptive PSO -> train
HASI-Net -> train baselines -> evaluate on the held-out test years -> write
figures + `results/summary_mp_<tag>.json`.


In [ ]:
summary_mp = run_experiment("mp", cfg, tag="mp")


In [ ]:
import pandas as pd, json
from IPython.display import Image, display
metrics = pd.read_csv(REPO / "results" / "metrics_mp.csv", index_col=0)
print("Test-set metrics (MP):"); print(metrics.round(4))

for fig in ["training_curves_mp.png", "pso_convergence_mp.png",
            "channel_weights_mp.png", "comparison_MAE_mp.png",
            "comparison_CSI_mp.png", "district_risk_mp.png",
            "pred_vs_actual_mp.png", "choropleth_mp.png"]:
    p = REPO / "results" / fig
    if p.exists():
        print("\n---", fig, "---"); display(Image(filename=str(p)))


## 3b. Component ablation
Isolates the contribution of each HASI-Net component (adaptive graph,
socioeconomic channel, spatial block, count-aware loss) by training short
variants with one component removed at a time.


In [ ]:
from hasi_net.experiment import run_ablation
abl = run_ablation("mp", cfg, tag="mp", epochs=40, verbose=False)
print(abl.round(4))
from IPython.display import Image, display
p = REPO / "results" / "ablation_mp.png"
if p.exists(): display(Image(filename=str(p)))


## 4. Benchmark — Chicago (fine-grained, exact lat/long)
Validates that HASI-Net also works on *precise* spatial data (objective 5 in the
proposal). Chicago incidents are pulled via the Socrata API and aggregated to a
community-area x month grid. We use a shorter window to keep the download
manageable.


In [ ]:
cfg_chi = cfg.override(chicago_year_start=2018, chicago_year_end=2022,
                         epochs=60, pso_iters=4, pso_particles=6, pso_swarms=2)
summary_chi = run_experiment("chicago", cfg_chi, tag="chicago")


## 5. Results summary & reproducibility
All artifacts are under `results/`: model weights (`hasi_net_<tag>.pt`),
metrics CSV, and a `summary_<tag>.json` capturing config + metrics + provenance.
The figures above are exactly those referenced in the paper.


In [ ]:
import json, pathlib
res = pathlib.Path(REPO / "results")
print("Artifacts:"); [print(" ", p.name, f"{p.stat().st_size//1024} KB") for p in sorted(res.glob("*"))]
print("\nMP summary:"); print(json.dumps({k: summary_mp[k] for k in
      ["dataset","device","n_nodes","n_years","panel_meta","channel_weights","best_val_mae"]}, indent=2))


## 6. Next steps
* Inspect `results/summary_mp.json` for the PSO-selected hyperparameters.
* The two research papers (methods + applied MP) are generated from these
  real outputs in `paper_a/` and `paper_b/`.
* To rerun with a different seed/horizon, edit `cfg` in the config cell and
  re-execute from section 3.
